# Assignment: Kaggle Competition - Plant Pathology 2021

**Competition:** [Plant Pathology 2021 - FGVC8](https://www.kaggle.com/competitions/plant-pathology-2021-fgvc8)  
**Task:** Multi-label classification of apple leaf diseases  
**Metric:** Mean F1-Score (higher is better)  

This notebook rebuilds the baseline pipeline in **PyTorch** around a **DINOv2-style Vision Transformer** using `timm`, then adds:

1. a stronger pretrained ViT backbone,
2. fixed-size full-image training/inference at `518x518`,
3. fixed per-class threshold at `0.5`,
4. selected-fold ensembling over test probabilities.


## Step 1: Imports

switch from TensorFlow/Keras to a PyTorch stack built around:

- `torch` for training and inference
- `timm` for DINOv2-style model loading
- `torchvision.transforms` for augmentation
- `sklearn` for multi-label encoding and macro F1 evaluation


In [ ]:
import os
import sys
import random
import socket
import subprocess
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"

try:
    import timm
except ImportError:
    subprocess.check_call(["python", "-m", "pip", "install", "-q", "timm"])
    import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
from torch import amp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import torchvision
from torchvision import transforms
from torchvision.transforms import InterpolationMode, v2

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    subprocess.check_call(["python", "-m", "pip", "install", "-q", "iterative-stratification"])
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("timm:", timm.__version__)
print("torchvision:", torchvision.__version__)
print("Iterative stratification ready")


## Step 2: Set Paths and Notebook Config

The notebook keeps the Kaggle-friendly structure, but adds a configuration block to control model choice, tiling, training budget, and a fixed threshold of `0.5` from one place.



In [ ]:
# Kaggle competition dataset paths
BASE_DIR = "/kaggle/input/competitions/plant-pathology-2021-fgvc8"
TRAIN_DIR = os.path.join(BASE_DIR, "train_images")
TEST_DIR = os.path.join(BASE_DIR, "test_images")

# Model and image settings
MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
FALLBACK_MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
IMAGE_SIZE = 518
NUM_FOLDS = 3
SELECTED_FOLDS = [0]
VALIDATION_MODE_NAME = "full_image"
USE_RANDAUG = True
RANDAUG_NUM_OPS = 2
RANDAUG_MAGNITUDE = 7

# Optimization settings
TRAIN_BATCH_SIZE = 32  # global batch size when DDP is enabled
EVAL_BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 2
NUM_EPOCHS_HEAD = 3
NUM_EPOCHS_FINETUNE = 3
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
FINETUNE_HEAD_LR = 5e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 2
SEED = 42
CHECKPOINT_DIR = "/kaggle/working"
CHECKPOINT_TEMPLATE = "best_dinov2_small_plant_pathology_fold{fold_id}.pth"
FOLD_ARTIFACT_TEMPLATE = "fold_artifact_{fold_id}.pth"
DDP_BACKEND = "nccl"
DDP_MASTER_ADDR = "127.0.0.1"
DDP_MASTER_PORT = None


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_GPUS = torch.cuda.device_count()
IN_NOTEBOOK = "ipykernel" in sys.modules
USE_DDP = DEVICE.type == "cuda" and NUM_GPUS > 1 and not IN_NOTEBOOK
USE_DATA_PARALLEL = DEVICE.type == "cuda" and NUM_GPUS > 1 and IN_NOTEBOOK
WORLD_SIZE = NUM_GPUS if USE_DDP else 1
NUM_WORKERS = 0 if IN_NOTEBOOK else DEFAULT_NUM_WORKERS
PIN_MEMORY = DEVICE.type == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2

print("Train dir:", TRAIN_DIR)
print("Test dir :", TEST_DIR)
print("Device   :", DEVICE)
print("Num GPUs :", NUM_GPUS)
print("In notebook:", IN_NOTEBOOK)
print("Use DDP  :", USE_DDP)
print("Use DataParallel:", USE_DATA_PARALLEL)
print("World size:", WORLD_SIZE)
print("Num folds:", NUM_FOLDS)
print("Selected folds:", SELECTED_FOLDS)
print("Train batch size (global):", TRAIN_BATCH_SIZE)
print("Eval image batch size:", EVAL_BATCH_SIZE)
print("Image size:", IMAGE_SIZE)
print("Eval mode:", VALIDATION_MODE_NAME)
print("RandAug enabled:", USE_RANDAUG)
print("RandAug ops/magnitude:", (RANDAUG_NUM_OPS, RANDAUG_MAGNITUDE))
print("Configured workers:", DEFAULT_NUM_WORKERS)
print("Active DataLoader workers:", NUM_WORKERS)


## Step 3: Load and Explore the Data

Load `train.csv` and `sample_submission.csv`, inspect the shape, and confirm the six disease classes needs to predict.


In [ ]:
train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
sample_df = pd.read_csv(os.path.join(BASE_DIR, "sample_submission.csv"))

ALL_LABELS = [
    "complex",
    "frog_eye_leaf_spot",
    "healthy",
    "powdery_mildew",
    "rust",
    "scab",
]
NUM_CLASSES = len(ALL_LABELS)

print("Train shape:", train_df.shape)
print("Sample submission shape:", sample_df.shape)
print("\nFirst 5 rows:")
print(train_df.head())
print("\nClasses:", ALL_LABELS)


In [ ]:
label_counts = train_df["labels"].value_counts().head(15)
print("Most common label strings:")
print(label_counts)


## Step 4: Show Sample Images

Before rebuilding the training stack, it helps to inspect raw apple leaf images. We keep training and inference consistent by using fixed-size full-image inputs.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (_, row) in zip(axes, train_df.sample(4, random_state=SEED).iterrows()):
    image = Image.open(os.path.join(TRAIN_DIR, row["image"])).convert("RGB")
    ax.imshow(image)
    ax.set_title(row["labels"], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## Step 5: Encode Labels and Create Multilabel Stratified Folds

This is still a multi-label problem, so convert the label strings into a 6-dimensional multi-hot target vector and build 3 stratified folds with preserved label balance.


In [ ]:
train_df["label_list"] = train_df["labels"].apply(lambda x: x.split())

mlb = MultiLabelBinarizer(classes=ALL_LABELS)
y_all = mlb.fit_transform(train_df["label_list"]).astype("float32")
image_names_all = train_df["image"].values
test_paths = sample_df["image"].values


def build_multilabel_folds(image_names, labels, n_splits, seed):
    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(train_idx, val_idx) for train_idx, val_idx in splitter.split(image_names, labels)]


FOLD_INDICES = build_multilabel_folds(image_names_all, y_all, NUM_FOLDS, SEED)
class_frequency = pd.Series(y_all.mean(axis=0), index=ALL_LABELS)

print("Label matrix shape:", y_all.shape)
print("\nOverall per-class positive rate:")
print(class_frequency.sort_values(ascending=False))
print(f"\nTest images: {len(test_paths)}")

fold_size_rows = []
fold_rate_table = pd.DataFrame({"overall_positive_rate": class_frequency})

for fold_id, (train_idx, val_idx) in enumerate(FOLD_INDICES):
    fold_size_rows.append(
        {
            "fold": fold_id,
            "train_size": len(train_idx),
            "val_size": len(val_idx),
        }
    )
    fold_rate_table[f"fold_{fold_id}_val_rate"] = y_all[val_idx].mean(axis=0)

fold_sizes_df = pd.DataFrame(fold_size_rows)
print("\nFold sizes:")
print(fold_sizes_df)
print("\nPer-class validation positive rate by fold:")
print(fold_rate_table.round(4))


## Step 6: Build the PyTorch Data Pipeline

The dataset uses a single whole-image mode for both training and inference data paths.

- `image`: standard whole-image training/evaluation/inference


In [ ]:
class PlantPathologyDataset(Dataset):
    def __init__(
        self,
        image_names,
        image_dir,
        labels=None,
        transform=None,
    ):
        self.image_names = list(image_names)
        self.image_dir = image_dir
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def _load_image(self, image_name):
        image_path = os.path.join(self.image_dir, image_name)
        with Image.open(image_path) as image:
            return image.convert("RGB")

    def _get_target(self, image_idx):
        if self.labels is None:
            return None
        return torch.tensor(self.labels[image_idx], dtype=torch.float32)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image = self._load_image(image_name)
        target = self._get_target(idx)

        image_tensor = self.transform(image) if self.transform else image
        if target is None:
            return image_tensor, image_name
        return image_tensor, target


def collate_train_images(batch):
    images = [item[0] for item in batch]
    targets = torch.stack([item[1] for item in batch])
    return images, targets


## Step 7: Define Augmentations and DataLoaders

follow the DINOv2/timm ecosystem by resolving the model's normalization config first, then build fold-specific DataLoaders. Training/validation/test all use fixed-size `518x518` inputs; only the training path applies stochastic augmentation.


In [ ]:
def select_model_name_and_data_config(primary_name, fallback_name):
    errors = {}

    for candidate in [primary_name, fallback_name]:
        try:
            probe_model = timm.create_model(candidate, pretrained=True, num_classes=0)
            data_config = timm.data.resolve_model_data_config(probe_model)
            del probe_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return candidate, data_config
        except Exception as exc:
            errors[candidate] = repr(exc)

    raise RuntimeError(f"Could not load a timm DINOv2 model. Errors: {errors}")


SELECTED_MODEL_NAME, DATA_CONFIG = select_model_name_and_data_config(
    MODEL_NAME,
    FALLBACK_MODEL_NAME,
)

NORMALIZE_MEAN = DATA_CONFIG.get("mean", (0.485, 0.456, 0.406))
NORMALIZE_STD = DATA_CONFIG.get("std", (0.229, 0.224, 0.225))


def build_train_image_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
        transforms.PILToTensor(),
    ])


train_image_transform = build_train_image_transform()

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])


def build_train_gpu_augmentor():
    transform_steps = [
        v2.RandomHorizontalFlip(p=0.5),
    ]
    if USE_RANDAUG:
        transform_steps.append(
            v2.RandAugment(
                num_ops=RANDAUG_NUM_OPS,
                magnitude=RANDAUG_MAGNITUDE,
                interpolation=InterpolationMode.BILINEAR,
            )
        )
    transform_steps.extend([
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ])
    return v2.Compose(transform_steps)


def prepare_train_batch(images, device, augmenter):
    augmenter.train()
    augmented = []

    for image in images:
        image = image.to(device, non_blocking=True).unsqueeze(0)
        image = augmenter(image)
        augmented.append(image.squeeze(0))

    batch = torch.stack(augmented, dim=0)
    return batch


def build_dataloader_kwargs(batch_size, shuffle=False, sampler=None, collate_fn=None):
    kwargs = {
        "batch_size": batch_size,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }
    if sampler is None:
        kwargs["shuffle"] = shuffle
    else:
        kwargs["sampler"] = sampler
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return kwargs


def create_dataloaders_for_fold(
    train_idx,
    val_idx,
    distributed_train=False,
    train_rank=0,
    train_world_size=1,
    build_eval_loaders=True,
    include_test=True,
):
    if distributed_train and TRAIN_BATCH_SIZE % train_world_size != 0:
        raise ValueError(
            f"TRAIN_BATCH_SIZE={TRAIN_BATCH_SIZE} must be divisible by train_world_size={train_world_size}."
        )

    train_dataset = PlantPathologyDataset(
        image_names=image_names_all[train_idx],
        image_dir=TRAIN_DIR,
        labels=y_all[train_idx],
        transform=train_image_transform,
    )

    train_sampler = None
    train_batch_size = TRAIN_BATCH_SIZE
    if distributed_train:
        train_sampler = DistributedSampler(
            train_dataset,
            num_replicas=train_world_size,
            rank=train_rank,
            shuffle=True,
            seed=SEED,
        )
        train_batch_size = TRAIN_BATCH_SIZE // train_world_size

    train_loader = DataLoader(
        train_dataset,
        **build_dataloader_kwargs(
            batch_size=train_batch_size,
            shuffle=train_sampler is None,
            sampler=train_sampler,
            collate_fn=collate_train_images,
        ),
    )

    loaders = {
        "train_loader": train_loader,
        "train_sampler": train_sampler,
    }

    if build_eval_loaders:
        val_dataset = PlantPathologyDataset(
            image_names=image_names_all[val_idx],
            image_dir=TRAIN_DIR,
            labels=y_all[val_idx],
            transform=eval_transform,
        )

        loaders["val_loader"] = DataLoader(
            val_dataset,
            **build_dataloader_kwargs(batch_size=EVAL_BATCH_SIZE, shuffle=False),
        )

    if include_test:
        test_image_dataset = PlantPathologyDataset(
            image_names=test_paths,
            image_dir=TEST_DIR,
            labels=None,
            transform=eval_transform,
        )

        loaders["test_image_loader"] = DataLoader(
            test_image_dataset,
            **build_dataloader_kwargs(batch_size=EVAL_BATCH_SIZE, shuffle=False),
        )

    return loaders


def tensor_to_display_image(tensor):
    image = tensor.permute(1, 2, 0).detach().cpu().numpy()
    image = image * np.array(NORMALIZE_STD)[None, None, :] + np.array(NORMALIZE_MEAN)[None, None, :]
    return np.clip(image, 0.0, 1.0)


preview_train_idx, preview_val_idx = FOLD_INDICES[0]
preview_loaders = create_dataloaders_for_fold(preview_train_idx, preview_val_idx, include_test=False)
preview_train_loader = preview_loaders["train_loader"]
preview_val_loader = preview_loaders["val_loader"]
preview_augmentor = build_train_gpu_augmentor().to(DEVICE)

preview_train_iter = iter(preview_train_loader)
sample_images, sample_targets = next(preview_train_iter)
del preview_train_iter
preview_augmented = prepare_train_batch([sample_images[0]], DEVICE, preview_augmentor)

preview_val_iter = iter(preview_val_loader)
val_images, val_targets = next(preview_val_iter)
del preview_val_iter

print("Selected timm model:", SELECTED_MODEL_NAME)
print("Normalization mean:", NORMALIZE_MEAN)
print("Normalization std :", NORMALIZE_STD)
print("Preview fold      : 0")
print("Train batch image count:", len(sample_images))
print("First raw train image tensor shape:", tuple(sample_images[0].shape))
print("First raw train image tensor dtype:", sample_images[0].dtype)
print("First augmented train image tensor shape:", tuple(preview_augmented[0].shape))
print("First augmented train image tensor dtype:", preview_augmented[0].dtype)
print("Train batch target tensor shape:", tuple(sample_targets.shape))
print("Validation image batch tensor shape:", tuple(val_images.shape))
print("Validation target tensor shape:", tuple(val_targets.shape))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for view_id, ax in enumerate(axes, start=1):
    aug_tensor = prepare_train_batch([sample_images[0]], DEVICE, preview_augmentor)[0]
    ax.imshow(tensor_to_display_image(aug_tensor))
    ax.set_title(f"Augmented view {view_id}")
    ax.axis("off")
plt.tight_layout()
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

del preview_loaders, preview_train_loader, preview_val_loader, preview_augmentor
del sample_images, sample_targets, preview_augmented, val_images, val_targets


## Step 8: Build the DINOv2 Model

Use `timm.create_model(...)` to build the ViT backbone and attach a multi-label classifier head on top.


In [ ]:
class PlantDiseaseDINOv2Model(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None)
        if self.feature_dim is None:
            raise ValueError("Could not infer backbone feature dimension.")

        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)

        if isinstance(features, dict):
            features = next(iter(features.values()))
        elif isinstance(features, (list, tuple)):
            features = features[0]

        if features.ndim == 3:
            features = features[:, 0]

        return self.head(features)


def unwrap_model(model):
    return model.module if isinstance(model, (DDP, nn.DataParallel)) else model


def freeze_backbone(model):
    actual_model = unwrap_model(model)
    for param in actual_model.backbone.parameters():
        param.requires_grad = False
    for param in actual_model.head.parameters():
        param.requires_grad = True


def unfreeze_last_blocks(model, num_blocks=2):
    freeze_backbone(model)
    actual_model = unwrap_model(model)

    if hasattr(actual_model.backbone, "blocks"):
        for block in actual_model.backbone.blocks[-num_blocks:]:
            for param in block.parameters():
                param.requires_grad = True

    if hasattr(actual_model.backbone, "norm"):
        for param in actual_model.backbone.norm.parameters():
            param.requires_grad = True


def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


preview_model = PlantDiseaseDINOv2Model(SELECTED_MODEL_NAME, NUM_CLASSES).to(DEVICE)
freeze_backbone(preview_model)

print(preview_model)
print("\nTrainable parameters after freezing backbone:", count_trainable_parameters(preview_model))

del preview_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Step 9: Define Fold Training Utilities

We package two-stage training, full-image validation, checkpointing, and selected-fold execution so the training loop stays easy to follow.


In [ ]:
def apply_thresholds(probabilities, thresholds):
    thresholds = np.asarray(thresholds, dtype=np.float32)
    predictions = (probabilities >= thresholds[None, :]).astype(np.int32)

    empty_rows = predictions.sum(axis=1) == 0
    if empty_rows.any():
        argmax_indices = probabilities[empty_rows].argmax(axis=1)
        predictions[empty_rows, :] = 0
        predictions[empty_rows, argmax_indices] = 1

    return predictions


def is_dist_ready():
    return dist.is_available() and dist.is_initialized()


def get_rank():
    return dist.get_rank() if is_dist_ready() else 0


def is_main_process():
    return get_rank() == 0


def dist_barrier():
    if is_dist_ready():
        dist.barrier()


def broadcast_python_object(obj, src=0):
    if not is_dist_ready():
        return obj
    payload = [obj if get_rank() == src else None]
    dist.broadcast_object_list(payload, src=src)
    return payload[0]


def reduce_loss_stats(total_loss, total_samples, device):
    stats = torch.tensor([total_loss, total_samples], dtype=torch.float64, device=device)
    if is_dist_ready():
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
    return float(stats[0].item()), int(stats[1].item())


def gather_epoch_arrays(local_probs, local_targets):
    if local_probs:
        local_probs = np.concatenate(local_probs, axis=0)
        local_targets = np.concatenate(local_targets, axis=0)
    else:
        local_probs = np.empty((0, NUM_CLASSES), dtype=np.float32)
        local_targets = np.empty((0, NUM_CLASSES), dtype=np.float32)

    if not is_dist_ready():
        return local_probs, local_targets

    gathered_probs = [None for _ in range(dist.get_world_size())]
    gathered_targets = [None for _ in range(dist.get_world_size())]
    dist.all_gather_object(gathered_probs, local_probs)
    dist.all_gather_object(gathered_targets, local_targets)

    if is_main_process():
        return np.concatenate(gathered_probs, axis=0), np.concatenate(gathered_targets, axis=0)
    return None, None


def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((DDP_MASTER_ADDR, 0))
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        return str(sock.getsockname()[1])


def build_checkpoint_path(fold_id):
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_TEMPLATE.format(fold_id=fold_id))


def build_fold_artifact_path(fold_id):
    return os.path.join(CHECKPOINT_DIR, FOLD_ARTIFACT_TEMPLATE.format(fold_id=fold_id))


def load_checkpoint_file(path, map_location):
    return torch.load(path, map_location=map_location, weights_only=True)


def load_training_artifact(path, map_location="cpu"):
    # Fold artifacts are produced by this notebook and include pandas/numpy objects.
    return torch.load(path, map_location=map_location, weights_only=False)


def train_one_epoch(model, loader, optimizer, criterion, device, scaler, train_gpu_augmentor):
    model.train()
    train_gpu_augmentor.train()
    total_loss = 0.0
    total_samples = 0
    local_probs = []
    local_targets = []

    progress = tqdm(loader, desc="Train", leave=False, disable=not is_main_process())
    for images, targets in progress:
        images = prepare_train_batch(images, device, train_gpu_augmentor)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        local_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        local_targets.append(targets.detach().cpu().numpy())
        if is_main_process():
            progress.set_postfix(loss=f"{loss.item():.4f}")

    total_loss, total_samples = reduce_loss_stats(total_loss, total_samples, device)
    all_probs, all_targets = gather_epoch_arrays(local_probs, local_targets)
    avg_loss = total_loss / max(total_samples, 1)

    macro_f1 = None
    if is_main_process():
        predictions = apply_thresholds(all_probs, np.full(NUM_CLASSES, 0.5, dtype=np.float32))
        macro_f1 = f1_score(all_targets, predictions, average="macro", zero_division=0)

    return avg_loss, macro_f1


def evaluate_image_level(model, loader, device, return_targets=True):
    model.eval()
    all_probabilities = []
    all_targets = []
    all_names = []

    with torch.no_grad():
        progress = tqdm(loader, desc="Image-level inference", leave=False)
        for batch in progress:
            if return_targets:
                images, targets = batch
                all_targets.append(targets.numpy())
            else:
                images, image_names = batch
                all_names.extend(list(image_names))

            images = images.to(device, non_blocking=True)
            with amp.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(images)

            probs = torch.sigmoid(logits).float().cpu().numpy()
            all_probabilities.append(probs)

    results = {
        "probs": np.concatenate(all_probabilities),
    }

    if return_targets:
        results["targets"] = np.concatenate(all_targets)
    if all_names:
        results["image_names"] = all_names

    return results


VALIDATION_CANDIDATE_METHODS = [VALIDATION_MODE_NAME]


def run_validation_protocol(model, loader, device, candidate_methods):
    if len(candidate_methods) != 1:
        raise ValueError(
            f"Expected exactly one validation mode, got {candidate_methods}. "
            "Set VALIDATION_CANDIDATE_METHODS to a single fixed method."
        )

    selected_method = candidate_methods[0]
    outputs = evaluate_image_level(
        model=model,
        loader=loader,
        device=device,
        return_targets=True,
    )

    val_probs = outputs["probs"]
    val_targets = outputs["targets"]
    predictions = apply_thresholds(val_probs, np.full(NUM_CLASSES, 0.5, dtype=np.float32))
    val_f1 = f1_score(val_targets, predictions, average="macro", zero_division=0)

    validation_results = {
        selected_method: {
            "probs": val_probs,
            "targets": val_targets,
            "val_f1": val_f1,
        }
    }

    return validation_results, selected_method, val_f1


def run_training_stage(
    model,
    train_loader,
    train_sampler,
    val_loader,
    optimizer,
    criterion,
    num_epochs,
    stage_name,
    checkpoint_path,
    candidate_methods,
    device,
    rank,
    train_gpu_augmentor,
    initial_best_f1=-np.inf,
):
    scaler = amp.GradScaler(device=device.type, enabled=USE_AMP)
    best_f1 = initial_best_f1
    patience_counter = 0
    history = []

    for epoch in range(1, num_epochs + 1):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        train_loss, train_f1 = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            scaler,
            train_gpu_augmentor,
        )

        epoch_state = None
        if rank == 0:
            validation_results, selected_method, val_f1 = run_validation_protocol(
                model=unwrap_model(model),
                loader=val_loader,
                device=device,
                candidate_methods=candidate_methods,
            )

            history_entry = {
                "stage": stage_name,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_f1": train_f1,
                "val_f1": val_f1,
                "val_validation_mode": selected_method,
            }
            history.append(history_entry)

            print(
                f"{stage_name} | Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss={train_loss:.4f} | train_f1={train_f1:.4f} | "
                f"val_image_f1={val_f1:.4f} | val_mode={selected_method}"
            )

            if val_f1 > best_f1:
                best_f1 = val_f1
                patience_counter = 0
                torch.save(
                    {
                        "model_state": unwrap_model(model).state_dict(),
                        "val_f1": val_f1,
                        "validation_mode": selected_method,
                        "stage": stage_name,
                        "epoch": epoch,
                    },
                    checkpoint_path,
                )
            else:
                patience_counter += 1

            should_stop = patience_counter >= PATIENCE
            if should_stop:
                print(f"Early stopping triggered during {stage_name}.")

            epoch_state = {
                "best_f1": float(best_f1),
                "should_stop": should_stop,
            }

        epoch_state = broadcast_python_object(epoch_state, src=0)
        best_f1 = epoch_state["best_f1"]
        if epoch_state["should_stop"]:
            break

    dist_barrier()
    best_checkpoint = load_checkpoint_file(checkpoint_path, map_location=device)
    unwrap_model(model).load_state_dict(best_checkpoint["model_state"])
    if rank == 0:
        print(
            f"Reloaded best checkpoint from stage: {best_checkpoint['stage']} "
            f"(val_f1={best_checkpoint['val_f1']:.4f}, mode={best_checkpoint.get('validation_mode', 'n/a')})"
        )
    dist_barrier()
    return pd.DataFrame(history) if rank == 0 else None, best_f1, best_checkpoint if rank == 0 else None


def ddp_worker(rank, world_size, fold_id, train_idx, val_idx):
    process_group_initialized = False
    distributed = world_size > 1
    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")

    try:
        set_seed(SEED + rank)
        if torch.cuda.is_available():
            torch.cuda.set_device(rank)

        if distributed:
            os.environ["MASTER_ADDR"] = DDP_MASTER_ADDR
            os.environ["MASTER_PORT"] = str(DDP_MASTER_PORT)
            os.environ["RANK"] = str(rank)
            os.environ["WORLD_SIZE"] = str(world_size)
            dist.init_process_group(backend=DDP_BACKEND, rank=rank, world_size=world_size)
            process_group_initialized = True

        loaders = create_dataloaders_for_fold(
            train_idx,
            val_idx,
            distributed_train=distributed,
            train_rank=rank,
            train_world_size=world_size,
            build_eval_loaders=(rank == 0),
            include_test=False,
        )
        train_loader = loaders["train_loader"]
        train_sampler = loaders["train_sampler"]
        val_loader = loaders.get("val_loader")
        checkpoint_path = build_checkpoint_path(fold_id)

        model = PlantDiseaseDINOv2Model(SELECTED_MODEL_NAME, NUM_CLASSES).to(device)
        if distributed:
            model = DDP(
                model,
                device_ids=[rank],
                output_device=rank,
                broadcast_buffers=False,
                find_unused_parameters=False,
            )
        elif USE_DATA_PARALLEL:
            model = nn.DataParallel(model)
        freeze_backbone(model)
        criterion = nn.BCEWithLogitsLoss()
        train_gpu_augmentor = build_train_gpu_augmentor().to(device)

        if rank == 0:
            print(f"\n{'=' * 12} Running Selected Fold {fold_id} {'=' * 12}")
            print("Checkpoint path:", checkpoint_path)
            print("Distributed training:", distributed)
            print("DataParallel training:", USE_DATA_PARALLEL)
            print("Global train batch size:", TRAIN_BATCH_SIZE)
            if distributed:
                print("Per-rank train batch size:", TRAIN_BATCH_SIZE // world_size)
            print("Eval image batch size:", EVAL_BATCH_SIZE)
            print("Trainable parameters after freezing backbone:", count_trainable_parameters(model))

        head_optimizer = torch.optim.AdamW(
            params=[param for param in model.parameters() if param.requires_grad],
            lr=HEAD_LR,
            weight_decay=WEIGHT_DECAY,
        )

        head_history, best_f1, best_checkpoint = run_training_stage(
            model=model,
            train_loader=train_loader,
            train_sampler=train_sampler,
            val_loader=val_loader,
            optimizer=head_optimizer,
            criterion=criterion,
            num_epochs=NUM_EPOCHS_HEAD,
            stage_name="Head training",
            checkpoint_path=checkpoint_path,
            candidate_methods=VALIDATION_CANDIDATE_METHODS,
            device=device,
            rank=rank,
            train_gpu_augmentor=train_gpu_augmentor,
        )

        unfreeze_last_blocks(model, num_blocks=2)
        actual_model = unwrap_model(model)
        backbone_params = [param for param in actual_model.backbone.parameters() if param.requires_grad]
        head_params = [param for param in actual_model.head.parameters() if param.requires_grad]

        if rank == 0:
            print("Trainable parameters after partial unfreeze:", count_trainable_parameters(model))
            print("Backbone parameter groups:", len(backbone_params))
            print("Head parameter groups    :", len(head_params))

        finetune_optimizer = torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": BACKBONE_LR},
                {"params": head_params, "lr": FINETUNE_HEAD_LR},
            ],
            weight_decay=WEIGHT_DECAY,
        )

        finetune_history, best_f1, best_checkpoint = run_training_stage(
            model=model,
            train_loader=train_loader,
            train_sampler=train_sampler,
            val_loader=val_loader,
            optimizer=finetune_optimizer,
            criterion=criterion,
            num_epochs=NUM_EPOCHS_FINETUNE,
            stage_name="Fine-tuning",
            checkpoint_path=checkpoint_path,
            candidate_methods=VALIDATION_CANDIDATE_METHODS,
            device=device,
            rank=rank,
            train_gpu_augmentor=train_gpu_augmentor,
            initial_best_f1=best_f1,
        )

        artifact = None
        if rank == 0:
            history_df = pd.concat([head_history, finetune_history], ignore_index=True)
            history_df["fold"] = fold_id

            validation_results, selected_method, best_val_f1 = run_validation_protocol(
                model=unwrap_model(model),
                loader=val_loader,
                device=device,
                candidate_methods=VALIDATION_CANDIDATE_METHODS,
            )

            artifact = {
                "fold": fold_id,
                "train_size": len(train_idx),
                "val_size": len(val_idx),
                "checkpoint_path": checkpoint_path,
                "best_stage": best_checkpoint["stage"],
                "best_epoch": int(best_checkpoint["epoch"]),
                "best_val_f1": float(best_val_f1),
                "history": history_df,
                "validation_results": validation_results,
            }
            torch.save(artifact, build_fold_artifact_path(fold_id))

        dist_barrier()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return artifact
    finally:
        if process_group_initialized:
            dist.destroy_process_group()


def run_fold_training(fold_id, train_idx, val_idx):
    artifact_path = build_fold_artifact_path(fold_id)
    checkpoint_path = build_checkpoint_path(fold_id)

    if os.path.exists(artifact_path):
        os.remove(artifact_path)
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    if USE_DDP and TRAIN_BATCH_SIZE % WORLD_SIZE != 0:
        raise ValueError(
            f"TRAIN_BATCH_SIZE={TRAIN_BATCH_SIZE} must be divisible by WORLD_SIZE={WORLD_SIZE}."
        )

    if USE_DDP:
        global DDP_MASTER_PORT
        DDP_MASTER_PORT = find_free_port()
        mp.start_processes(
            ddp_worker,
            args=(WORLD_SIZE, fold_id, train_idx, val_idx),
            nprocs=WORLD_SIZE,
            join=True,
            start_method="spawn",
        )
    else:
        ddp_worker(0, 1, fold_id, train_idx, val_idx)

    return load_training_artifact(artifact_path, map_location="cpu")


## Step 10: Run Selected Folds

Each selected fold runs the same two-stage schedule: head training, partial backbone unfreezing, and full-image validation at fixed threshold `0.5`.


In [ ]:
selected_fold_ids = sorted(set(SELECTED_FOLDS))
if not selected_fold_ids:
    raise ValueError("SELECTED_FOLDS must contain at least one fold id.")

invalid_fold_ids = [fold_id for fold_id in selected_fold_ids if fold_id < 0 or fold_id >= len(FOLD_INDICES)]
if invalid_fold_ids:
    raise ValueError(f"Invalid fold ids in SELECTED_FOLDS: {invalid_fold_ids}")

fold_artifacts = []
for fold_id in selected_fold_ids:
    train_idx, val_idx = FOLD_INDICES[fold_id]
    fold_artifacts.append(run_fold_training(fold_id, train_idx, val_idx))

FOLD_ARTIFACTS = {artifact["fold"]: artifact for artifact in fold_artifacts}
history_df = pd.concat([artifact["history"] for artifact in fold_artifacts], ignore_index=True)

fold_results_df = pd.DataFrame([
    {
        "fold": artifact["fold"],
        "train_size": artifact["train_size"],
        "val_size": artifact["val_size"],
        "best_stage": artifact["best_stage"],
        "best_epoch": artifact["best_epoch"],
        "best_val_f1": artifact["best_val_f1"],
        "checkpoint_path": artifact["checkpoint_path"],
    }
    for artifact in fold_artifacts
]).sort_values("fold").reset_index(drop=True)

method_results_rows = []
for artifact in fold_artifacts:
    metrics = artifact["validation_results"].get(VALIDATION_MODE_NAME)
    if metrics is None:
        raise KeyError(
            f"Expected validation mode '{VALIDATION_MODE_NAME}' in validation_results for fold {artifact['fold']}."
        )
    method_results_rows.append(
        {
            "fold": artifact["fold"],
            "validation_mode": VALIDATION_MODE_NAME,
            "val_f1": float(metrics["val_f1"]),
        }
    )

method_results_df = pd.DataFrame(method_results_rows)
method_summary_df = method_results_df.groupby("validation_mode", as_index=False).agg(
    mean_val_f1=("val_f1", "mean"),
    std_val_f1=("val_f1", "std"),
)
method_summary_df["std_val_f1"] = method_summary_df["std_val_f1"].fillna(0.0)
BEST_VAL_F1_MEAN = float(method_summary_df.loc[0, "mean_val_f1"])
BEST_VAL_F1_STD = float(method_summary_df.loc[0, "std_val_f1"])

print("Selected fold run finished.")
print()
print("Per-selected-fold summary:")
print(fold_results_df)
print()
print("Validation summary across selected folds:")
print(method_summary_df.round(4))
print()
print(f"Validation mode: {VALIDATION_MODE_NAME}")

fold_results_df


## Step 11: Summarize Selected-Fold Validation Results

The core metric is image-level F1 at fixed threshold `0.5` on the selected folds using the fixed full-image validation mode. We report per-fold scores and mean/std.


In [ ]:
if "method_summary_df" not in globals():
    if "FOLD_ARTIFACTS" not in globals():
        raise RuntimeError(
            "Selected-fold results are unavailable. Step 10 did not finish successfully. "
            "Rerun the fold-training cell before running this summary cell."
        )

    fold_artifacts = [FOLD_ARTIFACTS[fold_id] for fold_id in sorted(FOLD_ARTIFACTS)]
    method_results_rows = []
    for artifact in fold_artifacts:
        metrics = artifact["validation_results"].get(VALIDATION_MODE_NAME)
        if metrics is None:
            raise KeyError(
                f"Expected validation mode '{VALIDATION_MODE_NAME}' in validation_results for fold {artifact['fold']}."
            )
        method_results_rows.append(
            {
                "fold": artifact["fold"],
                "validation_mode": VALIDATION_MODE_NAME,
                "val_f1": float(metrics["val_f1"]),
            }
        )

    method_results_df = pd.DataFrame(method_results_rows)
    method_summary_df = method_results_df.groupby("validation_mode", as_index=False).agg(
        mean_val_f1=("val_f1", "mean"),
        std_val_f1=("val_f1", "std"),
    )
    method_summary_df["std_val_f1"] = method_summary_df["std_val_f1"].fillna(0.0)
    BEST_VAL_F1_MEAN = float(method_summary_df.loc[0, "mean_val_f1"])
    BEST_VAL_F1_STD = float(method_summary_df.loc[0, "std_val_f1"])

print("Validation summary across selected folds:")
print(method_summary_df.round(4))

fixed_validation_scores_df = method_results_df[["fold", "val_f1"]].copy()

print()
print(f"Validation mode: {VALIDATION_MODE_NAME}")
print(f"Mean F1: {BEST_VAL_F1_MEAN:.4f} +/- {BEST_VAL_F1_STD:.4f}")
print(f"Ensemble folds: {sorted(FOLD_ARTIFACTS)}")

fixed_validation_scores_df


## Step 12: Make Predictions on the Test Set

The test pipeline mirrors full-image validation, then ensembles all `SELECTED_FOLDS` checkpoints by averaging probabilities before decoding labels at threshold `0.5`.


In [ ]:
def probs_to_label(prob_row, thresholds):
    selected = [ALL_LABELS[i] for i, prob in enumerate(prob_row) if prob >= thresholds[i]]
    if not selected:
        selected = [ALL_LABELS[int(np.argmax(prob_row))]]
    return " ".join(selected)


if "FOLD_ARTIFACTS" not in globals() or not FOLD_ARTIFACTS:
    raise RuntimeError(
        "No trained fold artifacts found. Run Step 10 before Step 12."
    )

ensemble_fold_ids = sorted(FOLD_ARTIFACTS)
reference_fold_id = ensemble_fold_ids[0]
submission_train_idx, submission_val_idx = FOLD_INDICES[reference_fold_id]
submission_loaders = create_dataloaders_for_fold(submission_train_idx, submission_val_idx)
test_inference_loader = submission_loaders["test_image_loader"]

ensemble_probabilities = []
ensemble_image_names = None
checkpoint_stages = {}

for fold_id in ensemble_fold_ids:
    submission_model = PlantDiseaseDINOv2Model(SELECTED_MODEL_NAME, NUM_CLASSES).to(DEVICE)
    submission_checkpoint = load_checkpoint_file(
        FOLD_ARTIFACTS[fold_id]["checkpoint_path"],
        map_location=DEVICE,
    )
    unwrap_model(submission_model).load_state_dict(submission_checkpoint["model_state"])
    submission_model.eval()

    test_outputs = evaluate_image_level(
        model=submission_model,
        loader=test_inference_loader,
        device=DEVICE,
        return_targets=False,
    )

    fold_probabilities = test_outputs["probs"]
    fold_image_names = test_outputs.get("image_names", list(test_paths))

    if ensemble_image_names is None:
        ensemble_image_names = fold_image_names
    elif ensemble_image_names != fold_image_names:
        raise ValueError(
            f"Test image order mismatch between folds. Reference fold {reference_fold_id}, mismatched fold {fold_id}."
        )

    ensemble_probabilities.append(fold_probabilities)
    checkpoint_stages[fold_id] = submission_checkpoint.get("stage", "unknown")

if not ensemble_probabilities:
    raise RuntimeError("No fold predictions were generated for ensemble inference.")

test_probabilities = np.mean(np.stack(ensemble_probabilities, axis=0), axis=0)
test_image_names = ensemble_image_names
fixed_thresholds = np.full(NUM_CLASSES, 0.5, dtype=np.float32)
predicted_labels = [probs_to_label(prob_row, fixed_thresholds) for prob_row in test_probabilities]

print("Ensemble folds:", ensemble_fold_ids)
print("Checkpoint stages:", checkpoint_stages)
print("Evaluation mode:", VALIDATION_MODE_NAME)
print()
print("Sample predictions:")
for image_name, labels in list(zip(test_image_names, predicted_labels))[:5]:
    print(f"  {image_name} -> {labels}")


## Step 13: Create the Submission File

The submission format is unchanged:

- `image`: image filename
- `labels`: space-separated predicted labels


In [ ]:
submission = pd.DataFrame({
    "image": test_image_names,
    "labels": predicted_labels,
})

submission = submission.sort_values("image").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

print("submission.csv saved!")
print("\nFirst 5 rows:")
print(submission.head())
print("\nShape:", submission.shape)
print("Rows with empty labels:", int((submission["labels"].str.len() == 0).sum()))
